# Manual-linking pilot: Kaggle-only NER

Loads only selected original anonymized documents from the completed V3 retry output, runs the repository NER offset extractor on Kaggle, and writes `ner_predictions.jsonl`. No reconstruction or entity-linking outputs are used as labels.

In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import subprocess
import sys

# Deterministic 14-document pilot: 6 held-out hard cases, 5 automation rejects, 3 controls.
DOC_IDS = [
    '844545', '376624', '97505', '115249', '1438577', '59089',
    '1008960', '1424961', '1751329', '1143472', '560001',
    '900529', '758529', '387713',
]

input_root = Path('/kaggle/input')
clean_files = list(input_root.rglob('clean_10000.jsonl'))
challenge_files = list(input_root.rglob('challenge_rejected.jsonl'))
assert len(clean_files) == 1, f'Expected one V3 clean file; found {clean_files}'
assert len(challenge_files) == 1, f'Expected one V3 challenge file; found {challenge_files}'

records = {}
for path in clean_files:
    with path.open(encoding='utf-8-sig') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            doc_id = str(row.get('doc_name', row.get('doc_id', '')))
            if doc_id in DOC_IDS:
                text = row.get('original_anonymized_markdown')
                assert isinstance(text, str) and text, f'Missing original masked text: {doc_id}'
                records[doc_id] = {'doc_id': doc_id, 'text': text}
for path in challenge_files:
    with path.open(encoding='utf-8-sig') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            doc_id = str(row.get('doc_id', ''))
            if doc_id in DOC_IDS:
                text = row.get('source_markdown')
                assert isinstance(text, str) and text, f'Missing original masked text: {doc_id}'
                records[doc_id] = {'doc_id': doc_id, 'text': text}
missing = set(DOC_IDS) - set(records)
assert not missing, f'Selected documents missing from V3 output: {sorted(missing)}'
rows = [records[doc_id] for doc_id in DOC_IDS]
print(f'Prepared {len(rows)} original masked documents; chars={sum(len(r["text"]) for r in rows):,}')

# Clone only public source code. No local environment or model cache is used.
repo = Path('/kaggle/working/VDT-Anonymization')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/no1ceboy/VDT-Anonymization.git', str(repo)], check=True)
sys.path.insert(0, str(repo / 'src'))
from vdt_anonymization.pipeline.ner import choose_device, infer_document, load_model, model_label_map, parse_entity_types
import torch

device = choose_device('auto')
model_args = Namespace(model_source='huggingface', model_name='NlpHUST/ner-vietnamese-electra-base', model_path=None, local_files_only=False, dtype='fp16')
tokenizer, model, model_reference = load_model(model_args, device)
label_map = model_label_map(model)
entity_types = parse_entity_types('PER,LOC,ORG')
print('Device:', device, '| model:', model_reference, '| labels:', label_map)

output_path = Path('/kaggle/working/ner_predictions.jsonl')
with output_path.open('w', encoding='utf-8', newline='\n') as output:
    for index, row in enumerate(rows, 1):
        entities = infer_document(
            text=row['text'], tokenizer=tokenizer, model=model, device=device,
            label_map=label_map, entity_types=entity_types, max_length=512,
            stride=128, batch_size=16, min_confidence=0.0,
        )
        output.write(json.dumps({
            'doc_id': row['doc_id'], 'char_len': len(row['text']),
            'model': model_reference, 'entities': entities,
        }, ensure_ascii=False) + '\n')
        output.flush()
        print(f'{index}/{len(rows)} {row["doc_id"]}: {len(entities)} NER spans')
print('Saved:', output_path, '| bytes:', output_path.stat().st_size)